# Linajea Biohub: 8 train / 2 validation

Thí nghiệm này chọn 8 sample từ embryo 44b6 để train và 2 sample từ embryo 6bba để validation. Hai embryo không xuất hiện đồng thời ở hai nhóm. Dùng toàn bộ annotation GEFF hiện có trên 8 sample train, không hide node. Biohub GT vẫn là sparse; 100% GT ở đây nghĩa là 100% các nhãn đã được cung cấp.

Sau train, notebook predict toàn bộ 100 frame của hai sample validation, xuất graph GEFF, tính known-cell/edge recall ở 2, 4 và 7 µm, rồi tính tracking metric bằng code của ban tổ chức. Selection constant và track cost cố định ở 7/7 trước khi đọc GT validation. Không dùng validation GT để train hoặc chọn ILP. Output ở /kaggle/working/biohub_holdout_8x2_reverse. Đây là chiều ngược với notebook train 6bba → validation 44b6; so sánh hai kết quả để đánh giá độ ổn định theo embryo.

Để so sánh công bằng với chiều trước, giữ nguyên checkpoint settings, ILP 7/7 và đặt PINNED_COMMIT bằng commit đã dùng ở lần chạy trước.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display

DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
ROOT = Path('/kaggle/working/biohub_holdout_8x2_reverse')
REPO = Path('/kaggle/working/linajea')
BRANCH = 'biohub-io'
PINNED_COMMIT = None  # Điền SHA commit đã push để cố định phiên bản code.

TRAIN_EMBRYO = '44b6'
VAL_EMBRYO = '6bba'
SPLIT_SEED = 42
TRAIN_SEED = 42
ITERATIONS = 16000
CHECKPOINT_STRIDE = 2000
SELECTION_CONSTANT = 7.0
TRACK_COST = 7.0
SCORE_THRESHOLD = 0.2
MAX_CANDIDATES_PER_FRAME = 200

SPLIT = ROOT / 'split.json'
TRAIN = ROOT / 'train'
VALIDATION = ROOT / 'validation'
ROOT.mkdir(parents=True, exist_ok=True)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)
print('Train embryo:', TRAIN_EMBRYO, 'Validation embryo:', VAL_EMBRYO)
print('Output:', ROOT)


## 1. Cài đặt mã nguồn và dependency

Kaggle notebook này dùng Internet cho mục đích nghiên cứu/validation. Để tái lập chính xác, điền PINNED_COMMIT sau khi push thay đổi repo.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU trong Kaggle Notebook Settings')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
if PINNED_COMMIT:
    subprocess.run(['git', '-C', str(REPO), 'checkout', PINNED_COMMIT], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'origin/' + BRANCH], check=True)
COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(ROOT / 'code_commit.txt').write_text(COMMIT + '\n')
for required in ('run_scripts/biohub_holdout_split.py',
                 'run_scripts/train_biohub_multi.py',
                 'run_scripts/validate_biohub_multi.py'):
    if not (REPO / required).is_file():
        raise FileNotFoundError('Push code mới lên fork trước: ' + required)
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *map(str, args)], check=True)
pip('-r', REPO / 'requirements-biohub-io.txt')
pip('-r', REPO / 'requirements-biohub-train.txt')
pip('--no-deps', 'git+https://github.com/abred/daisy@ab7b82054126a50ab4ad5dcf2ef885b02d33aab8')
pip('--no-deps', 'git+https://github.com/Kainmueller-Lab/funlib.learn.torch@36ef666d16bddda0bf3146d4c0efbe9b95c8463d')
pip('-e', REPO, '--no-deps')
print('Linajea commit:', COMMIT, 'GPU:', torch.cuda.get_device_name(0))


## 2. Tạo split 8/2 trước khi train

Danh sách sample được chọn bằng seed 42 từ hai embryo khác nhau và lưu trong split.json. Nếu chạy lại, giữ nguyên file này.


In [ ]:
if not SPLIT.exists():
    subprocess.run([
        sys.executable, '-m', 'run_scripts.biohub_holdout_split',
        '--data-dir', str(DATA_DIR), '--output', str(SPLIT),
        '--train-embryo', TRAIN_EMBRYO, '--val-embryo', VAL_EMBRYO,
        '--seed', str(SPLIT_SEED)
    ], cwd=REPO, check=True)
split = json.loads(SPLIT.read_text())
assert len(split['train']) == 8 and len(split['validation']) == 2
assert {s.split('_')[0] for s in split['train']} == {TRAIN_EMBRYO}
assert {s.split('_')[0] for s in split['validation']} == {VAL_EMBRYO}
print('Train:', split['train'])
print('Validation:', split['validation'])
display(pd.DataFrame.from_dict(split['sample_details'], orient='index'))


## 3. Train từ đầu trên 8 sample

16.000 iteration chia ngẫu nhiên giữa 8 nguồn dữ liệu; mỗi iteration chỉ lấy một ROI nhỏ, không phải đọc toàn bộ 8 ảnh một lần. Checkpoint lưu mỗi 2.000 iteration. Không đưa 2 sample validation vào config train.


In [ ]:
def run_logged(command, log_path, show_every=500):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO,
                                   stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            if 'Batch: iteration=' in line:
                iteration = int(line.split('Batch: iteration=')[1].split(',')[0])
                if iteration == 1 or iteration % show_every == 0:
                    print(line.rstrip(), flush=True)
            elif any(word in line for word in
                     ('Full inference:', 'Traceback', 'Error',
                      'Predicted GEFF directory:', 'Training complete:')):
                print(line.rstrip(), flush=True)
        if process.wait() != 0:
            raise RuntimeError(f'Command failed; read {log_path}')

checkpoint = TRAIN / f'train_net_checkpoint_{ITERATIONS}'
if not checkpoint.is_file():
    run_logged([
        sys.executable, '-m', 'run_scripts.train_biohub_multi',
        '--data-dir', str(DATA_DIR), '--split', str(SPLIT),
        '--output-dir', str(TRAIN), '--iterations', str(ITERATIONS),
        '--checkpoint-stride', str(CHECKPOINT_STRIDE),
        '--seed', str(TRAIN_SEED)
    ], TRAIN / 'train.log')
print('Checkpoint:', checkpoint, 'bytes:', checkpoint.stat().st_size)
config_path = TRAIN / 'config.toml'
config_text = config_path.read_text()
for name in split['validation']:
    assert name not in config_text, f'Validation leakage: {name}'
print('Train config excludes both validation samples')


## 4. Predict hai sample validation và đánh giá nhãn sparse

Suy luận toàn bộ hai sample, solve ILP fixed 7/7, rồi lưu GEFF và bảng recall. GT validation chỉ được dùng sau khi graph đã chọn xong. Prediction cache cho từng sample được dùng lại khi chạy lại cell.

Hai cột candidate_node_recall và candidate_edge_recall đo graph trước ILP; known_node_recall và known_edge_recall đo graph đã chọn sau ILP. Mỗi stage được match một-một với GT ở 7 µm.
Candidate edge recall dùng cùng phép match một-một nhưng match lại độc lập cho từng graph; nếu có nhiều tâm gần một GT, số edge match có thể thay đổi khi ILP loại node.


In [ ]:
run_logged([
    sys.executable, '-m', 'run_scripts.validate_biohub_multi',
    '--data-dir', str(DATA_DIR), '--split', str(SPLIT),
    '--config', str(config_path), '--checkpoint', str(checkpoint),
    '--output-dir', str(VALIDATION),
    '--selection-constant', str(SELECTION_CONSTANT),
    '--track-cost', str(TRACK_COST),
    '--score-threshold', str(SCORE_THRESHOLD),
    '--max-candidates-per-frame', str(MAX_CANDIDATES_PER_FRAME),
    '--device', 'cuda'
], VALIDATION / 'run.log', show_every=500)

local_metrics = pd.read_csv(VALIDATION / 'summary_7um.csv')
print('CELL DETECTION — trước và sau ILP, match 7 µm')
display(local_metrics[['sample', 'candidate_nodes', 'predicted_nodes',
                       'candidate_matched_gt_nodes', 'matched_gt_nodes',
                       'known_gt_nodes', 'candidate_node_recall',
                       'known_node_recall',
                       'predictions_unmatched_by_sparse_gt']])
print('TEMPORAL EDGES — trước và sau ILP, match 7 µm')
display(local_metrics[['sample', 'candidate_edges', 'predicted_edges',
                       'candidate_matched_gt_edges', 'matched_gt_edges',
                       'known_gt_edges', 'candidate_edge_recall',
                       'known_edge_recall', 'conditional_edge_recall']])
print('Predicted GEFF:', VALIDATION / 'predictions')


## 5. Tracking metric chính thức của ban tổ chức

Dùng bản commit cố định của royerlab/kaggle-cell-tracking-competition. Metric match centroid trong bán kính 7 µm, tính edge Jaccard đã điều chỉnh theo số node dự đoán và cộng 0,1 lần division Jaccard. Score có thể lớn hơn 1. Các prediction không khớp sparse GT không được tự động gọi là false cell.


In [ ]:
OFFICIAL_COMMIT = '075fc5f5a52d11077f9dc2b074644618f26939e2'
OFFICIAL_REPO = Path('/kaggle/working/kaggle-cell-tracking-competition')
if not OFFICIAL_REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/royerlab/kaggle-cell-tracking-competition.git',
                    str(OFFICIAL_REPO)], check=True)
subprocess.run(['git', '-C', str(OFFICIAL_REPO), 'checkout', OFFICIAL_COMMIT],
               check=True)
# Cài metric sau train/inference để dependency mới không thay đổi môi trường model.
try:
    import tracksdata, geff, polars
except ImportError:
    pip('git+https://github.com/royerlab/tracksdata@e13cf379b5127deeb8301ce56410fda35b5a3cf9')
    import tracksdata, geff, polars
from importlib.metadata import version
print('Metric dependencies:', 'tracksdata', version('tracksdata'),
      'geff', version('geff'), 'polars', version('polars'))
sys.path.insert(0, str(OFFICIAL_REPO / 'src'))
sys.path.insert(0, str(OFFICIAL_REPO / 'scripts'))
from evaluate import evaluate_pairs
from tracking_cellmot.metrics import summarise

official_rows, skipped = evaluate_pairs(
    VALIDATION / 'predictions', DATA_DIR, max_distance=7.0)
if skipped or len(official_rows) != 2:
    raise RuntimeError(f'Official metric skipped samples: {skipped}')
official_summary = summarise(official_rows)
pd.DataFrame([{'sample': name, **row}
              for name, row in zip(sorted(split['validation']), official_rows)]
            ).to_csv(VALIDATION / 'official_per_sample.csv', index=False)
(VALIDATION / 'official_summary.json').write_text(
    json.dumps(official_summary, indent=2, default=float))
print('Official tracking metric:')
display(pd.DataFrame([official_summary]))
print('Saved:', VALIDATION / 'official_summary.json')
